In [ ]:
#1. Import Library
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("Matplotlib:", plt.matplotlib.__version__)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Pandas: 3.0.5
Matplotlib: 3.11.2


In [79]:
#2. Load Model Bahasa Indonesia
import spacy_stanza

nlp = spacy_stanza.load_pipeline(
    "id",
    processors="tokenize,pos,lemma,depparse",
    use_gpu=False
)

print("Model bahasa Indonesia berhasil dimuat.")

2026-10-17 23:00:15 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-10-17 23:00:16 INFO: Downloaded file to C:\Users\acer\AppData\Local\StanfordNLP\stanza\Cache\1.15.0\resources\resources.json
2026-10-17 23:00:16 WARNING: Language id package default expects mwt, which has been added
2026-10-17 23:00:18 INFO: Loading these models for language: id (Indonesian):
| Processor | Package      |
----------------------------
| tokenize  | gsd_nocharlm |
| mwt       | gsd          |
| pos       | gsd_charlm   |
| lemma     | gsd_nocharlm |
| depparse  | gsd_charlm   |

2026-10-17 23:00:18 INFO: Using device: cpu
2026-10-17 23:00:18 INFO: Loading: tokenize
2026-10-17 23:00:18 INFO: Loading: mwt
2026-10-17 23:00:18 INFO: Loading: pos
2026-10-17 23:00:25 INFO: Loading: lemma
2026-10-17 23:00:25 INFO: Loading: depparse
2026-10-17 23:00:25 INFO: Done loading processors!


Model bahasa Indonesia berhasil dimuat.


In [44]:
# Uji Model
teks = "Mahasiswa sedang mempelajari pemrosesan bahasa alami."
doc = nlp(teks)

# Validasi berdasarkan hasil pemrosesan
assert len(doc) > 0, "Tidak ada token yang dihasilkan."
assert any(token.pos_ for token in doc), "POS belum tersedia."
assert any(token.dep_ for token in doc), "Dependency belum tersedia."

print("Token berhasil dibuat:", len(doc))
print("POS tagging tersedia: Ya")
print("Dependency parsing tersedia: Ya")
print("\nHasil analisis:")

for token in doc:
    print(
        f"{token.text:15} | "
        f"{token.pos_:8} | "
        f"{token.lemma_:12} | "
        f"{token.dep_:10} | "
        f"Kepala: {token.head.text}"
    )

Token berhasil dibuat: 7
POS tagging tersedia: Ya
Dependency parsing tersedia: Ya

Hasil analisis:
Mahasiswa       | PROPN    | mahasiswa    | nsubj      | Kepala: mempelajari
sedang          | AUX      | sedang       | aux        | Kepala: mempelajari
mempelajari     | VERB     | pelajar      | root       | Kepala: mempelajari
pemrosesan      | NOUN     | proses       | obj        | Kepala: mempelajari
bahasa          | NOUN     | bahasa       | compound   | Kepala: pemrosesan
alami           | ADJ      | alami        | amod       | Kepala: bahasa
.               | PUNCT    | .            | punct      | Kepala: mempelajari


In [45]:
#3. Memproses Kalimat
sentence = "Mahasiswa belajar Natural Language Processing."
doc = nlp(sentence)

print("Kalimat:", sentence)
print("\nToken hasil analisis:")
for token in doc:
    print(f"{token.text:<20} POS={token.pos_:<10} Dependency={token.dep_}")

Kalimat: Mahasiswa belajar Natural Language Processing.

Token hasil analisis:
Mahasiswa            POS=PROPN      Dependency=nsubj
belajar              POS=VERB       Dependency=root
Natural              POS=PROPN      Dependency=obj
Language             POS=PROPN      Dependency=flat:name
Processing           POS=PROPN      Dependency=flat:name
.                    POS=PUNCT      Dependency=punct


In [46]:
#4. Analisis POS Token
doc = nlp("Mahasiswa belajar Natural Language Processing.")

print(f"{'Token':<18} {'POS':<12} {'Lemma'}")
print("-" * 55)
for token in doc:
    print(f"{token.text:<18} {token.pos_:<12} {token.lemma_}")

Token              POS          Lemma
-------------------------------------------------------
Mahasiswa          PROPN        mahasiswa
belajar            VERB         belajar
Natural            PROPN        natural
Language           PROPN        language
Processing         PROPN        processing
.                  PUNCT        .


In [47]:
#5. Membuat Tabel Analisis POS
def analyze_pos(text, nlp_model=nlp):
    doc = nlp_model(str(text))
    return pd.DataFrame([{
        "Token": token.text,
        "POS": token.pos_,
        "POS Tag": token.tag_,
        "Lemma": token.lemma_,
        "Dependency": token.dep_,
        "Head": token.head.text
    } for token in doc])

df_pos = analyze_pos("Mahasiswa belajar NLP.")
display(df_pos)

,Token,POS,POS Tag,Lemma,Dependency,Head
0,Mahasiswa,PROPN,NSD,mahasiswa,nsubj,belajar
1,belajar,VERB,NSD,belajar,root,belajar
2,NLP,PROPN,X--,nlp,obj,belajar
3,.,PUNCT,Z--,.,punct,belajar


In [48]:
#6. Analisis Beberapa Kalimat
sentences = [
    "Mahasiswa belajar NLP.",
    "Dosen mengajar Natural Language Processing.",
    "Mahasiswa membaca buku.",
    "Sistem memproses data teks.",
    "Model bahasa memprediksi kata berikutnya."
]

for sentence in sentences:
    print(f"\nKalimat: {sentence}")
    df_result = analyze_pos(sentence)
    display(df_result[["Token", "POS", "Dependency", "Head"]])


Kalimat: Mahasiswa belajar NLP.


,Token,POS,Dependency,Head
0,Mahasiswa,PROPN,nsubj,belajar
1,belajar,VERB,root,belajar
2,NLP,PROPN,obj,belajar
3,.,PUNCT,punct,belajar



Kalimat: Dosen mengajar Natural Language Processing.


,Token,POS,Dependency,Head
0,Dosen,PROPN,nsubj,mengajar
1,mengajar,VERB,root,mengajar
2,Natural,PROPN,obj,mengajar
3,Language,PROPN,flat:name,Natural
4,Processing,PROPN,flat:name,Language
5,.,PUNCT,punct,mengajar



Kalimat: Mahasiswa membaca buku.


,Token,POS,Dependency,Head
0,Mahasiswa,PROPN,nsubj,membaca
1,membaca,VERB,root,membaca
2,buku,NOUN,obj,membaca
3,.,PUNCT,punct,membaca



Kalimat: Sistem memproses data teks.


,Token,POS,Dependency,Head
0,Sistem,PROPN,nsubj,memproses
1,memproses,VERB,root,memproses
2,data,NOUN,obj,memproses
3,teks,NOUN,compound,data
4,.,PUNCT,punct,memproses



Kalimat: Model bahasa memprediksi kata berikutnya.


,Token,POS,Dependency,Head
0,Model,NOUN,nsubj,memprediksi
1,bahasa,NOUN,compound,Model
2,memprediksi,VERB,root,memprediksi
3,kata,NOUN,obj,memprediksi
4,berikutnya,ADJ,amod,kata
5,.,PUNCT,punct,memprediksi


In [49]:
#7. Dependency Parsing
doc = nlp("Mahasiswa belajar NLP.")

print(f"{'Token':<18} {'Dependency':<15} {'Head'}")
print("-" * 50)
for token in doc:
    print(f"{token.text:<18} {token.dep_:<15} {token.head.text}")

Token              Dependency      Head
--------------------------------------------------
Mahasiswa          nsubj           belajar
belajar            root            belajar
NLP                obj             belajar
.                  punct           belajar


In [50]:
#8. Visualisasi Dependency
from spacy import displacy

doc = nlp("Mahasiswa belajar Natural Language Processing.")
displacy.render(doc, style="dep", jupyter=True, options={"distance": 100})

In [ ]:
# Simpan dependency tree sebagai HTML
from pathlib import Path
from spacy import displacy

html_path = Path("dependency_tree.html")

# Simpan visualisasi dependency sebagai HTML
with html_path.open("w", encoding="utf-8") as f:
    displacy.render(
        doc,
        style="dep",
        page=True,
        jupyter=False
    )

print("Lokasi file:", html_path.resolve())
print("File berhasil dibuat:", html_path.exists())



Lokasi file: C:\Users\acer\Documents\NLP\dependency_tree.html
File berhasil dibuat: True


In [ ]:
from pathlib import Path
from spacy import displacy

# Buat HTML diagram dependency tree
html_content = displacy.render(
    doc,
    style="dep",
    page=True,
    jupyter=False
)

# Simpan HTML ke file
html_path = Path("dependency_tree.html")
html_path.write_text(html_content, encoding="utf-8")

print("Diagram dependency tree berhasil disimpan!")
print("Lokasi:", html_path.resolve())
print("Ukuran file:", html_path.stat().st_size, "byte")


Diagram dependency tree berhasil disimpan!
Lokasi: C:\Users\acer\Documents\NLP\dependency_tree.html
Ukuran file: 4309 byte


In [64]:
#9. Analisis Dataset 
DATA_PATHS = [
    Path("../dataset/dataset_preprocessed.csv"),
    Path("dataset_preprocessed.csv"),
    Path("../dataset_preprocessed.csv"),
]

data_path = next((path for path in DATA_PATHS if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "File dataset_preprocessed.csv tidak ditemukan. Simpan di folder dataset "
        "pada root proyek atau sesuaikan DATA_PATHS."
    )

df = pd.read_csv(data_path)
print("Dataset:", data_path.resolve())
print("Ukuran dataset:", df.shape)
print("Daftar kolom:", df.columns.tolist())
display(df.head())

Dataset: C:\Users\acer\Documents\NLP\dataset_preprocessed.csv
Ukuran dataset: (220051, 14)
Daftar kolom: ['video_id', 'comment_id', 'parent_comment_id', 'level', 'username', 'nickname', 'comment', 'create_time', 'lowercase_text', 'clean_text', 'tokens', 'tokens_no_stopword', 'tokens_preprocessed', 'processed_text']


,video_id,comment_id,parent_comment_id,level,username,nickname,comment,create_time,lowercase_text,clean_text,tokens,tokens_no_stopword,tokens_preprocessed,processed_text
0,7.526460e+18,7.643300e+18,NaN,0.0,jbm123jayabetonmandiri,Christian sudartio,Beliau bilang untung nya 2 m ?,5/24/2026 10:58,beliau bilang untung nya 2 m ?,beliau bilang untung nya m,"['beliau', 'bilang', 'untung', 'nya', 'm']","['beliau', 'bilang', 'untung', 'nya', 'm']","['beliau', 'bilang', 'untung', 'nya', 'm']",beliau bilang untung nya m
1,7.526460e+18,7.643360e+18,7.643300e+18,1.0,sahrulmustakimm,sahrul,Makasih Mas,5/24/2026 14:47,makasih mas,makasih mas,"['makasih', 'mas']","['makasih', 'mas']","['makasih', 'mas']",makasih mas
2,7.526460e+18,7.643370e+18,7.643300e+18,1.0,jbm123jayabetonmandiri,Christian sudartio,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,5/24/2026 15:13,gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya m barang yg di jual ajh ga ada m,"['gimana', 'untung', 'nya', 'mbarang', 'yg', 'di', 'jual', 'ajh', 'ga', 'ada', 'm', '😁']","['gimana', 'untung', 'nya', 'mbarang', 'yg', 'jual', 'ajh', 'ga', 'm', '😁']","['gimana', 'untung', 'nya', 'm', 'barang', 'yg', 'jual', 'ajh', 'ga', 'm']",gimana untung nya m barang yg jual ajh ga m
3,7.526460e+18,7.643370e+18,7.643300e+18,1.0,sahrulmustakimm,sahrul,[Sticker] dibilang Makasih Mas udah 2M itu,5/24/2026 15:35,[sticker] dibilang makasih mas udah 2m itu,sticker dibilang makasih mas udah m itu,"['sticker', 'dibilang', 'makasih', 'mas', 'udah', 'm', 'itu']","['sticker', 'dibilang', 'makasih', 'mas', 'udah', 'm']","['sticker', 'bilang', 'makasih', 'mas', 'udah', 'm']",sticker bilang makasih mas udah m
4,7.526460e+18,7.643370e+18,7.643300e+18,1.0,jbm123jayabetonmandiri,Christian sudartio,😳😳😳,5/24/2026 15:12,😳😳😳,NaN,['😳😳😳'],['😳😳😳'],[],NaN


In [65]:
if "comment" in df.columns:
    TEXT_COLUMN = "comment"
elif "text" in df.columns:
    TEXT_COLUMN = "text"
elif "processed_text" in df.columns:
    TEXT_COLUMN = "processed_text"
else:
    raise ValueError("Kolom teks tidak ditemukan: " + str(df.columns.tolist()))

text_df = df[df[TEXT_COLUMN].notna()].copy()
text_df[TEXT_COLUMN] = text_df[TEXT_COLUMN].astype(str).str.strip()
text_df = text_df[text_df[TEXT_COLUMN] != ""]

print("Kolom teks:", TEXT_COLUMN)
print("Jumlah baris dataset:", len(df))
print("Jumlah teks valid:", len(text_df))
print("Jumlah teks kosong:", df[TEXT_COLUMN].isna().sum())

Kolom teks: comment
Jumlah baris dataset: 220051
Jumlah teks valid: 219747
Jumlah teks kosong: 304


In [66]:
# Sampel 10 komentar yang konsisten dan dapat diulang
sample = text_df.sample(n=min(10, len(text_df)), random_state=42)

for number, (_, row) in enumerate(sample.iterrows(), start=1):
    text = str(row[TEXT_COLUMN])[:200]
    doc = nlp(text)
    pos_counts = {}
    for token in doc:
        pos = token.pos_ or "UNK"
        pos_counts[pos] = pos_counts.get(pos, 0) + 1
    print(f"\nSampel {number}: {text[:100]}")
    print("Distribusi POS:", pos_counts)


Sampel 1: mungkin maksudnya distop..tidak ada pembangunan baru untuk indomart&alfamart,bukan yg ada ditutup,
Distribusi POS: {'AUX': 1, 'VERB': 4, 'DET': 1, 'NOUN': 3, 'PUNCT': 4, 'PART': 2, 'ADJ': 1, 'ADP': 1, 'PRON': 1}

Sampel 2: betul nek ditru trus ke koyok di Idak Idak
Distribusi POS: {'ADJ': 1, 'X': 1, 'NOUN': 3, 'ADP': 1, 'VERB': 1, 'PROPN': 2}

Sampel 3: udah ga jaman koprasi
Distribusi POS: {'X': 1, 'PART': 1, 'NOUN': 2}

Sampel 4: masih percaya walk in ?
Distribusi POS: {'ADV': 1, 'VERB': 1, 'PROPN': 2, 'PUNCT': 1}

Sampel 5: Ya Allah gini amat Indonesia
Distribusi POS: {'PART': 1, 'PROPN': 3, 'VERB': 1}

Sampel 6: Jyujyur tidakkkk
Distribusi POS: {'PROPN': 2}

Sampel 7: program Bpk Presiden Sagat baik.👍
cuman pak,realisasinya ke masyarakat gk sesuai.
Distribusi POS: {'NOUN': 5, 'PROPN': 3, 'ADJ': 2, 'PUNCT': 4, 'SPACE': 1, 'VERB': 1, 'PRON': 1, 'ADP': 1}

Sampel 8: maaf kaya gudang...
Distribusi POS: {'VERB': 1, 'ADJ': 1, 'NOUN': 1, 'PUNCT': 2}

Sampel 9: korupnya ngorup b

In [67]:
#10. Analisis Kesalahan
informal_texts = [
    "aku lg belajar NLP nih",
    "gue udah baca buku itu",
    "modelnya bagus bgt sih",
    "nggak ngerti sama sekali",
    "udah instal belum?"
]

print("Analisis teks informal")
print("=" * 70)
for text in informal_texts:
    doc = nlp(text)
    print(f"\nTeks: {text}")
    for token in doc:
        print(
            f"  {token.text:<15} POS={token.pos_:<10} "
            f"Dependency={token.dep_:<12} Head={token.head.text}"
        )

Analisis teks informal

Teks: aku lg belajar NLP nih
  aku             POS=PRON       Dependency=nsubj        Head=lg
  lg              POS=VERB       Dependency=root         Head=lg
  belajar         POS=VERB       Dependency=xcomp        Head=lg
  NLP             POS=PROPN      Dependency=obj          Head=belajar
  nih             POS=NOUN       Dependency=compound     Head=NLP

Teks: gue udah baca buku itu
  gue             POS=PROPN      Dependency=nsubj        Head=baca
  udah            POS=ADV        Dependency=advmod       Head=baca
  baca            POS=VERB       Dependency=root         Head=baca
  buku            POS=NOUN       Dependency=obj          Head=baca
  itu             POS=DET        Dependency=det          Head=buku

Teks: modelnya bagus bgt sih
  model           POS=NOUN       Dependency=root         Head=model
  nya             POS=PRON       Dependency=nmod:poss    Head=model
  bagus           POS=ADJ        Dependency=amod         Head=model
  bgt            

In [73]:
import pandas as pd
from pathlib import Path

data_path = Path.cwd() / "dataset_preprocessed.csv"

df = pd.read_csv(data_path)

print("Dataset berhasil dimuat!")
print("Lokasi:", data_path)
print("Ukuran dataset:", df.shape)
print("Nama kolom:", df.columns.tolist())

display(df.head())

Dataset berhasil dimuat!
Lokasi: c:\Users\acer\Documents\NLP\dataset_preprocessed.csv
Ukuran dataset: (220051, 14)
Nama kolom: ['video_id', 'comment_id', 'parent_comment_id', 'level', 'username', 'nickname', 'comment', 'create_time', 'lowercase_text', 'clean_text', 'tokens', 'tokens_no_stopword', 'tokens_preprocessed', 'processed_text']


,video_id,comment_id,parent_comment_id,level,username,nickname,comment,create_time,lowercase_text,clean_text,tokens,tokens_no_stopword,tokens_preprocessed,processed_text
0,7.526460e+18,7.643300e+18,NaN,0.0,jbm123jayabetonmandiri,Christian sudartio,Beliau bilang untung nya 2 m ?,5/24/2026 10:58,beliau bilang untung nya 2 m ?,beliau bilang untung nya m,"['beliau', 'bilang', 'untung', 'nya', 'm']","['beliau', 'bilang', 'untung', 'nya', 'm']","['beliau', 'bilang', 'untung', 'nya', 'm']",beliau bilang untung nya m
1,7.526460e+18,7.643360e+18,7.643300e+18,1.0,sahrulmustakimm,sahrul,Makasih Mas,5/24/2026 14:47,makasih mas,makasih mas,"['makasih', 'mas']","['makasih', 'mas']","['makasih', 'mas']",makasih mas
2,7.526460e+18,7.643370e+18,7.643300e+18,1.0,jbm123jayabetonmandiri,Christian sudartio,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,5/24/2026 15:13,gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya m barang yg di jual ajh ga ada m,"['gimana', 'untung', 'nya', 'mbarang', 'yg', 'di', 'jual', 'ajh', 'ga', 'ada', 'm', '😁']","['gimana', 'untung', 'nya', 'mbarang', 'yg', 'jual', 'ajh', 'ga', 'm', '😁']","['gimana', 'untung', 'nya', 'm', 'barang', 'yg', 'jual', 'ajh', 'ga', 'm']",gimana untung nya m barang yg jual ajh ga m
3,7.526460e+18,7.643370e+18,7.643300e+18,1.0,sahrulmustakimm,sahrul,[Sticker] dibilang Makasih Mas udah 2M itu,5/24/2026 15:35,[sticker] dibilang makasih mas udah 2m itu,sticker dibilang makasih mas udah m itu,"['sticker', 'dibilang', 'makasih', 'mas', 'udah', 'm', 'itu']","['sticker', 'dibilang', 'makasih', 'mas', 'udah', 'm']","['sticker', 'bilang', 'makasih', 'mas', 'udah', 'm']",sticker bilang makasih mas udah m
4,7.526460e+18,7.643370e+18,7.643300e+18,1.0,jbm123jayabetonmandiri,Christian sudartio,😳😳😳,5/24/2026 15:12,😳😳😳,NaN,['😳😳😳'],['😳😳😳'],[],NaN


In [74]:
text_df = df["comment"].dropna().astype(str).str.strip()

# Pilih teks yang cukup panjang dan mengandung huruf
text_df = text_df[
    (text_df.str.len() >= 20)
    & text_df.str.contains(r"[A-Za-zÀ-ÿ]", regex=True)
]

sample_10 = text_df.sample(
    n=10,
    random_state=42
)

for i, teks in enumerate(sample_10, start=1):
    print(f"{i}. {teks}")

1. ini konsepnya Indonesia mau dimundurkan bukan majuinnn? segala mau nutup indo alfa demi ini 😭
2. Kayanya lebih adem indomaret + pegawainya cantik
3. gk tertarik banget belanja disitu
4. planogram nya kacau, padahal rak samping bisa buat sewa supplier lumayan pemasukan toko
5. pelanggan setia warung Madura
6. jagoan windah nata barang nya
7. dia gak tau apa unilever sama wings paling gak mau di jadiin 1 rak 😭
8. AHAHAH CARA NGEDISPLAY NYA NGAKAK BGT 😭😭😭
9. ktp n KK..untuk di siapkan 2029 pilpres.
10. [Sticker] tanpa planogram


In [ ]:
#Tugas Analisis Linguistik
 #10 kalimat dari dataset.
import pandas as pd
import spacy_stanza
from pathlib import Path
from spacy import displacy
from IPython.display import display, IFrame

# Memuat model bahasa Indonesia
nlp = spacy_stanza.load_pipeline(
    "id",
    processors="tokenize,pos,lemma,depparse",
    use_gpu=False
)

kalimat_uji = [
    "ini konsepnya Indonesia mau dimundurkan bukan majuinnn? segala mau nutup indo alfa demi ini 😭",
    "Kayanya lebih adem indomaret + pegawainya cantik",
    "gk tertarik banget belanja disitu",
    "planogram nya kacau, padahal rak samping bisa buat sewa supplier lumayan pemasukan toko",
    "pelanggan setia warung Madura",
    "jagoan windah nata barang nya",
    "dia gak tau apa unilever sama wings paling gak mau di jadiin 1 rak 😭",
    "AHAHAH CARA NGEDISPLAY NYA NGAKAK BGT 😭😭😭",
    "ktp n KK..untuk di siapkan 2029 pilpres.",
    "[Sticker] tanpa planogram"
]

hasil = []
dokumen = []

for nomor, kalimat in enumerate(kalimat_uji, start=1):
    doc = nlp(kalimat)
    dokumen.append(doc)

    token_list = []
    pos_list = []
    dependency_list = []

    for token in doc:
        token_list.append(token.text)
        pos_list.append(f"{token.text}: {token.pos_}")
        dependency_list.append(
            f"{token.text} → {token.head.text} ({token.dep_})"
        )

    hasil.append({
        "No": nomor,
        "Kalimat": kalimat,
        "Token": ", ".join(token_list),
        "POS": "; ".join(pos_list),
        "Dependency": "; ".join(dependency_list)
    })

df_hasil = pd.DataFrame(hasil)

# Menampilkan tabel hasil analisis
pd.set_option("display.max_colwidth", 100)
display(df_hasil)


2026-10-17 21:54:10 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-10-17 21:54:11 INFO: Downloaded file to C:\Users\acer\AppData\Local\StanfordNLP\stanza\Cache\1.15.0\resources\resources.json
2026-10-17 21:54:11 WARNING: Language id package default expects mwt, which has been added
2026-10-17 21:54:13 INFO: Loading these models for language: id (Indonesian):
| Processor | Package      |
----------------------------
| tokenize  | gsd_nocharlm |
| mwt       | gsd          |
| pos       | gsd_charlm   |
| lemma     | gsd_nocharlm |
| depparse  | gsd_charlm   |

2026-10-17 21:54:13 INFO: Using device: cpu
2026-10-17 21:54:13 INFO: Loading: tokenize
2026-10-17 21:54:13 INFO: Loading: mwt
2026-10-17 21:54:13 INFO: Loading: pos
2026-10-17 21:54:17 INFO: Loading: lemma
2026-10-17 21:54:18 INFO: Loading: depparse
2026-10-17 21:54:18 INFO: Done loading processors!


,No,Kalimat,Token,POS,Dependency
0,1,ini konsepnya Indonesia mau dimundurkan bukan majuinnn? segala mau nutup indo alfa demi ini 😭,"ini, konsep, nya, Indonesia, mau, dimundurkan, bukan, majuinnn, ?, segala, mau, nutup, indo, alf...",ini: DET; konsep: NOUN; nya: PRON; Indonesia: PROPN; mau: AUX; dimundurkan: VERB; bukan: PART; m...,ini → konsep (det); konsep → dimundurkan (nsubj:pass); nya → konsep (nmod:poss); Indonesia → kon...
1,2,Kayanya lebih adem indomaret + pegawainya cantik,"Kaya, nya, lebih, adem, indomaret, +, pegawai, nya, cantik",Kaya: ADJ; nya: PRON; lebih: ADV; adem: ADJ; indomaret: NOUN; +: SYM; pegawai: NOUN; nya: PRON; ...,Kaya → Kaya (root); nya → Kaya (nmod:poss); lebih → adem (advmod); adem → Kaya (amod); indomaret...
2,3,gk tertarik banget belanja disitu,"gk, tertarik, banget, belanja, disitu",gk: PROPN; tertarik: VERB; banget: X; belanja: VERB; disitu: ADP,gk → tertarik (nsubj:pass); tertarik → tertarik (root); banget → tertarik (goeswith); belanja → ...
3,4,"planogram nya kacau, padahal rak samping bisa buat sewa supplier lumayan pemasukan toko","planogram, nya, kacau, ,, padahal, rak, samping, bisa, buat, sewa, supplier, lumayan, pemasukan,...","planogram: NOUN; nya: PRON; kacau: ADJ; ,: PUNCT; padahal: SCONJ; rak: NOUN; samping: NOUN; bisa...","planogram → planogram (root); nya → planogram (nmod:poss); kacau → planogram (amod); , → sewa (p..."
4,5,pelanggan setia warung Madura,"pelanggan, setia, warung, Madura",pelanggan: NOUN; setia: ADJ; warung: NOUN; Madura: PROPN,pelanggan → pelanggan (root); setia → pelanggan (amod); warung → pelanggan (compound); Madura → ...
5,6,jagoan windah nata barang nya,"jagoan, windah, nata, barang, nya",jagoan: NOUN; windah: NOUN; nata: NOUN; barang: NOUN; nya: PRON,jagoan → jagoan (root); windah → jagoan (compound); nata → windah (compound); barang → nata (com...
6,7,dia gak tau apa unilever sama wings paling gak mau di jadiin 1 rak 😭,"dia, gak, tau, apa, unilever, sama, wings, paling, gak, mau, di, jadiin, 1, rak, 😭",dia: PRON; gak: PART; tau: VERB; apa: ADV; unilever: NOUN; sama: ADJ; wings: PROPN; paling: ADV;...,dia → tau (nsubj); gak → tau (advmod); tau → tau (root); apa → mau (advmod); unilever → mau (nsu...
7,8,AHAHAH CARA NGEDISPLAY NYA NGAKAK BGT 😭😭😭,"AHAHAH, CARA, NGEDISPLAY, NYA, NGAKAK, BGT, 😭😭, 😭",AHAHAH: PROPN; CARA: PROPN; NGEDISPLAY: PROPN; NYA: PRON; NGAKAK: PROPN; BGT: PROPN; 😭😭: PROPN; ...,AHAHAH → AHAHAH (root); CARA → AHAHAH (flat:name); NGEDISPLAY → CARA (flat:name); NYA → NGEDISPL...
8,9,ktp n KK..untuk di siapkan 2029 pilpres.,"ktp, n, KK, ., .untuk, di, siapkan, 2029, pilpres, .",ktp: NOUN; n: PROPN; KK: PROPN; .: PUNCT; .untuk: SCONJ; di: VERB; siapkan: X; 2029: NUM; pilpre...,ktp → di (nsubj:pass); n → ktp (nmod); KK → n (flat:name); . → ktp (punct); .untuk → di (mark); ...
9,10,[Sticker] tanpa planogram,"[, Sticker, ], tanpa, planogram",[: PUNCT; Sticker: PROPN; ]: PUNCT; tanpa: ADP; planogram: NOUN,[ → Sticker (punct); Sticker → Sticker (root); ] → Sticker (punct); tanpa → planogram (case); pl...


In [76]:
#Visualisasi dependency tree untuk setiap kalimat

output_dir = Path.cwd() / "hasil_dependency"
output_dir.mkdir(parents=True, exist_ok=True)

for nomor, (kalimat, doc) in enumerate(
    zip(kalimat_uji, dokumen), start=1
):
    html = displacy.render(
        doc,
        style="dep",
        page=True,
        jupyter=False,
        options={"compact": False, "distance": 100}
    )

    file_html = output_dir / f"dependency_tree_{nomor:02d}.html"
    file_html.write_text(html, encoding="utf-8")

    print(f"{nomor}. {file_html.name}")

print("\nSemua visualisasi selesai dibuat.")
print("Folder:", output_dir)


1. dependency_tree_01.html
2. dependency_tree_02.html
3. dependency_tree_03.html
4. dependency_tree_04.html
5. dependency_tree_05.html
6. dependency_tree_06.html
7. dependency_tree_07.html
8. dependency_tree_08.html
9. dependency_tree_09.html
10. dependency_tree_10.html

Semua visualisasi selesai dibuat.
Folder: c:\Users\acer\Documents\NLP\hasil_dependency


In [78]:

display(df_hasil)


,No,Kalimat,Token,POS,Dependency
0,1,ini konsepnya Indonesia mau dimundurkan bukan majuinnn? segala mau nutup indo alfa demi ini 😭,"ini, konsep, nya, Indonesia, mau, dimundurkan, bukan, majuinnn, ?, segala, mau, nutup, indo, alf...",ini: DET; konsep: NOUN; nya: PRON; Indonesia: PROPN; mau: AUX; dimundurkan: VERB; bukan: PART; m...,ini → konsep (det); konsep → dimundurkan (nsubj:pass); nya → konsep (nmod:poss); Indonesia → kon...
1,2,Kayanya lebih adem indomaret + pegawainya cantik,"Kaya, nya, lebih, adem, indomaret, +, pegawai, nya, cantik",Kaya: ADJ; nya: PRON; lebih: ADV; adem: ADJ; indomaret: NOUN; +: SYM; pegawai: NOUN; nya: PRON; ...,Kaya → Kaya (root); nya → Kaya (nmod:poss); lebih → adem (advmod); adem → Kaya (amod); indomaret...
2,3,gk tertarik banget belanja disitu,"gk, tertarik, banget, belanja, disitu",gk: PROPN; tertarik: VERB; banget: X; belanja: VERB; disitu: ADP,gk → tertarik (nsubj:pass); tertarik → tertarik (root); banget → tertarik (goeswith); belanja → ...
3,4,"planogram nya kacau, padahal rak samping bisa buat sewa supplier lumayan pemasukan toko","planogram, nya, kacau, ,, padahal, rak, samping, bisa, buat, sewa, supplier, lumayan, pemasukan,...","planogram: NOUN; nya: PRON; kacau: ADJ; ,: PUNCT; padahal: SCONJ; rak: NOUN; samping: NOUN; bisa...","planogram → planogram (root); nya → planogram (nmod:poss); kacau → planogram (amod); , → sewa (p..."
4,5,pelanggan setia warung Madura,"pelanggan, setia, warung, Madura",pelanggan: NOUN; setia: ADJ; warung: NOUN; Madura: PROPN,pelanggan → pelanggan (root); setia → pelanggan (amod); warung → pelanggan (compound); Madura → ...
5,6,jagoan windah nata barang nya,"jagoan, windah, nata, barang, nya",jagoan: NOUN; windah: NOUN; nata: NOUN; barang: NOUN; nya: PRON,jagoan → jagoan (root); windah → jagoan (compound); nata → windah (compound); barang → nata (com...
6,7,dia gak tau apa unilever sama wings paling gak mau di jadiin 1 rak 😭,"dia, gak, tau, apa, unilever, sama, wings, paling, gak, mau, di, jadiin, 1, rak, 😭",dia: PRON; gak: PART; tau: VERB; apa: ADV; unilever: NOUN; sama: ADJ; wings: PROPN; paling: ADV;...,dia → tau (nsubj); gak → tau (advmod); tau → tau (root); apa → mau (advmod); unilever → mau (nsu...
7,8,AHAHAH CARA NGEDISPLAY NYA NGAKAK BGT 😭😭😭,"AHAHAH, CARA, NGEDISPLAY, NYA, NGAKAK, BGT, 😭😭, 😭",AHAHAH: PROPN; CARA: PROPN; NGEDISPLAY: PROPN; NYA: PRON; NGAKAK: PROPN; BGT: PROPN; 😭😭: PROPN; ...,AHAHAH → AHAHAH (root); CARA → AHAHAH (flat:name); NGEDISPLAY → CARA (flat:name); NYA → NGEDISPL...
8,9,ktp n KK..untuk di siapkan 2029 pilpres.,"ktp, n, KK, ., .untuk, di, siapkan, 2029, pilpres, .",ktp: NOUN; n: PROPN; KK: PROPN; .: PUNCT; .untuk: SCONJ; di: VERB; siapkan: X; 2029: NUM; pilpre...,ktp → di (nsubj:pass); n → ktp (nmod); KK → n (flat:name); . → ktp (punct); .untuk → di (mark); ...
9,10,[Sticker] tanpa planogram,"[, Sticker, ], tanpa, planogram",[: PUNCT; Sticker: PROPN; ]: PUNCT; tanpa: ADP; planogram: NOUN,[ → Sticker (punct); Sticker → Sticker (root); ] → Sticker (punct); tanpa → planogram (case); pl...


In [81]:
# Visualisasi Dependency Tree di Dalam Notebook
from IPython.display import display, HTML, Markdown
from spacy import displacy

for nomor, (kalimat, doc) in enumerate(zip(kalimat_uji, dokumen), start=1):
    display(Markdown(f"#### Dependency Tree {nomor}: {kalimat}"))
    html_tree = displacy.render(
        doc,
        style="dep",
        jupyter=False,
        options={"compact": True, "distance": 80}
    )
    display(HTML(html_tree))


#### Dependency Tree 1: ini konsepnya Indonesia mau dimundurkan bukan majuinnn? segala mau nutup indo alfa demi ini 😭

#### Dependency Tree 2: Kayanya lebih adem indomaret + pegawainya cantik

#### Dependency Tree 3: gk tertarik banget belanja disitu

#### Dependency Tree 4: planogram nya kacau, padahal rak samping bisa buat sewa supplier lumayan pemasukan toko

#### Dependency Tree 5: pelanggan setia warung Madura

#### Dependency Tree 6: jagoan windah nata barang nya

#### Dependency Tree 7: dia gak tau apa unilever sama wings paling gak mau di jadiin 1 rak 😭

#### Dependency Tree 8: AHAHAH CARA NGEDISPLAY NYA NGAKAK BGT 😭😭😭

#### Dependency Tree 9: ktp n KK..untuk di siapkan 2029 pilpres.

#### Dependency Tree 10: [Sticker] tanpa planogram

In [83]:
#Tabel Pemeriksaan Kesalahan (catatan pemeriksaan pada 10 kalimat yang dianalisis)
catatan_pemeriksaan = [
    "Bahasa informal, pemanjangan kata, dan emoji; perlu verifikasi manual.",
    "Ejaan informal 'Kayanya' dan nama tempat/merek; perlu verifikasi manual.",
    "Bentuk tidak baku 'gk' dan 'disitu'; perlu verifikasi manual.",
    "Kata gabungan/tidak baku 'planogram nya' serta istilah supplier; perlu verifikasi manual.",
    "Frasa singkat tanpa konteks kalimat lengkap; perlu verifikasi manual.",
    "Susunan kata informal dan nama orang/kanal; perlu verifikasi manual.",
    "Bahasa informal, ejaan tidak baku, serta nama merek; perlu verifikasi manual.",
    "Huruf kapital berlebihan, slang, dan emoji; perlu verifikasi manual.",
    "Singkatan 'ktp', 'n', dan 'KK' serta struktur kalimat; perlu verifikasi manual.",
    "Teks penanda '[Sticker]' dan frasa singkat; perlu verifikasi manual."
]

df_hasil_lengkap = df_hasil.copy()
df_hasil_lengkap["Pemeriksaan Kesalahan"] = catatan_pemeriksaan[:len(df_hasil_lengkap)]
display(df_hasil_lengkap[["No", "Kalimat", "Pemeriksaan Kesalahan"]])

,No,Kalimat,Pemeriksaan Kesalahan
0,1,ini konsepnya Indonesia mau dimundurkan bukan majuinnn? segala mau nutup indo alfa demi ini 😭,"Bahasa informal, pemanjangan kata, dan emoji; perlu verifikasi manual."
1,2,Kayanya lebih adem indomaret + pegawainya cantik,Ejaan informal 'Kayanya' dan nama tempat/merek; perlu verifikasi manual.
2,3,gk tertarik banget belanja disitu,Bentuk tidak baku 'gk' dan 'disitu'; perlu verifikasi manual.
3,4,"planogram nya kacau, padahal rak samping bisa buat sewa supplier lumayan pemasukan toko",Kata gabungan/tidak baku 'planogram nya' serta istilah supplier; perlu verifikasi manual.
4,5,pelanggan setia warung Madura,Frasa singkat tanpa konteks kalimat lengkap; perlu verifikasi manual.
5,6,jagoan windah nata barang nya,Susunan kata informal dan nama orang/kanal; perlu verifikasi manual.
6,7,dia gak tau apa unilever sama wings paling gak mau di jadiin 1 rak 😭,"Bahasa informal, ejaan tidak baku, serta nama merek; perlu verifikasi manual."
7,8,AHAHAH CARA NGEDISPLAY NYA NGAKAK BGT 😭😭😭,"Huruf kapital berlebihan, slang, dan emoji; perlu verifikasi manual."
8,9,ktp n KK..untuk di siapkan 2029 pilpres.,"Singkatan 'ktp', 'n', dan 'KK' serta struktur kalimat; perlu verifikasi manual."
9,10,[Sticker] tanpa planogram,Teks penanda '[Sticker]' dan frasa singkat; perlu verifikasi manual.
